In [18]:
import sys
print(sys.executable)

d:\Database\.venv\Scripts\python.exe


In [19]:
import os

os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["hadoop.home.dir"] = "C:\\hadoop"

Task 1: Data Ingestion & Exploration
Objective


Objective
Load the dataset into PySpark and perform exploratory analysis.


Requirements

Create Spark Session


In [20]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
    .appName("DataAnalysis")\
    .getOrCreate()

Load DataSet

In [21]:
df=spark.read.csv("BNPParibas_Data.csv",header=True,inferSchema=True)

Display:


Schema

In [22]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- support_tickets: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- churn: integer (nullable = true)



Record Count


In [23]:
df.count()

1000

Null Count


In [24]:
from pyspark.sql.functions import col,when,count
df.select([count(
    when(col(c).isNull(),c)
    ).alias(c)
    for c in df.columns
    ]).show()


+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges|contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|          0|  0|            0|              0|            0|            0|               0|              0|             0|    0|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+



Duplicate Count


In [25]:
duplicate_count=df.count()-df.distinct().count()
duplicate_count

0

Data Types

In [26]:
df.dtypes

[('customer_id', 'int'),
 ('age', 'int'),
 ('tenure_months', 'int'),
 ('monthly_charges', 'double'),
 ('total_charges', 'double'),
 ('contract_type', 'string'),
 ('internet_service', 'string'),
 ('support_tickets', 'int'),
 ('payment_method', 'string'),
 ('churn', 'int')]

Task 2: ETL Pipeline Development

Objective
Build a complete ETL pipeline.


Extract
Read source dataset.


In [27]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
    .appName("ELT_Pipeline")\
    .getOrCreate()

df=spark.read.csv("BNPParibas_Data.csv",header=True,inferSchema=True)

Transform

Perform:



Missing Value Treatment


In [28]:
from pyspark.sql.functions import col,when,count
df.select([count(
    when(col(c).isNull(),c)
    ).alias(c)
    for c in df.columns
    ]).show()

+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges|contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|          0|  0|            0|              0|            0|            0|               0|              0|             0|    0|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+



In [29]:
df=df.fillna({"internet_service":"Unknown"})

Duplicate Removal

In [30]:
df=df.drop_duplicates()

Data Type Conversion

In [31]:
df.dtypes  # already correct

[('customer_id', 'int'),
 ('age', 'int'),
 ('tenure_months', 'int'),
 ('monthly_charges', 'double'),
 ('total_charges', 'double'),
 ('contract_type', 'string'),
 ('internet_service', 'string'),
 ('support_tickets', 'int'),
 ('payment_method', 'string'),
 ('churn', 'int')]

Feature Engineering

Average monthly spend

In [38]:
df.withColumn("average_monthly_spend",col("total_charges")/col("tenure_months"))
df.show(5)

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|average_monthly_spend|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|   61.974666666666664|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|    85.44131578947368|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|   28.330750000000002|
|        410| 19|           51|          88.61|      4368.39|      One Year|      

Aggregation

Find avg monthly charges 

In [33]:
from pyspark.sql.functions import avg
df.groupBy("internet_service")\
.agg(avg("monthly_charges"))

DataFrame[internet_service: string, avg(monthly_charges): double]

Load:
Store transformed data in:


In [37]:
df.write.mode("overwrite").parquet("silver/")

Py4JJavaError: An error occurred while calling o485.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: Hadoop home directory C:\hadoop does not exist -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.m

Task 3: ELT Pipeline & Medallion Architecture
